# Caso HyM · Detección de Data Drift con métodos estadísticos y ADWIN

**Plantilla colaborativa, ejecutable y orientada a agentes · Google Colab · Python 3**

**Equipo (5):** Marco (coordinación e integración), Poma (preparación), Vera (KS/PSI), Aldana (chi-cuadrado y drift controlado), Brickzon (ADWIN).

**Propósito académico:** estudiar si las distribuciones de características de clientes difieren entre cohortes de incorporación, y contrastar pruebas estadísticas con detección secuencial ADWIN.

> **Identidad del dataset.** El CSV y los notebooks facilitados se refieren a `marketing_campaign.csv` (campaña de marketing). Aunque el trabajo se llama «Caso HyM», este archivo **no demuestra por sí solo** que los registros procedan de H&M. No describir resultados como hechos empíricos de H&M sin documentación externa.

> **Estado de esta entrega:** plantilla con base funcional y espacios de interpretación. **No contiene resultados reales inventados.** Para obtenerlos hace falta cargar el CSV. Los integrantes deberán verificar, enriquecer, ejecutar e interpretar las subsecciones asignadas.

### Índice y responsables

| Sección | Contenido | Encargado |
|:--|:--|:--|
| 1 | Contexto y objetivos | Marco |
| 2 | Entorno y parámetros | Marco |
| 3 | Carga y auditoría inicial | Poma |
| 4 | Calidad y preprocesamiento | Poma |
| 5 | Cohortes de referencia/comparación | Marco |
| 6 | Análisis exploratorio | Poma |
| 7 | Kolmogorov-Smirnov | Vera |
| 8 | Population Stability Index | Vera |
| 9 | Chi-cuadrado | Aldana |
| 10 | ADWIN | Brickzon |
| 11 | Drift controlado | Aldana, con interfaces de Vera y Brickzon |
| 12 | Consolidación y revisión | Marco |
| 13 | Conclusiones y referencias | Marco |

**Ruta crítica:** secciones 2–5 → métodos 7–10 en paralelo → sección 11 → integración 12–13.

**Convención:** una celda de texto bajo **cada subsección** indica el responsable, el alcance, los contratos y la definición de terminado.

## 1. Introducción, alcance y objetivos

**Responsable principal:** Marco. **Objetivo:** delimitar qué significa Data Drift en este dataset.

### 1.1. Pregunta de investigación

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Explicar diferencias entre cohortes de clientes según `Dt_Customer`; formular H₀/H₁ de KS y chi-cuadrado, y precisar que ADWIN analiza secuencias de observaciones, no etiquetas de clase.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Texto final sin atribuciones no justificadas a H&M y con limitaciones temporales explícitas.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Pregunta:** ¿varían las distribuciones de ingresos, consumo, canales de compra y categorías de clientes entre las cohortes más antiguas y más recientes?

**Objetivo general:** detectar, cuantificar e interpretar señales de cambio en características de clientes mediante KS, PSI, chi-cuadrado y ADWIN.

**Objetivos específicos:** preparar los datos de forma reproducible; construir cohortes temporales sin solapamiento; comparar variables apropiadas; examinar ADWIN sobre secuencias ordenadas; validar sensibilidad en un experimento controlado; discutir coincidencias, discordancias y limitaciones.

**Restricción de validez:** `Dt_Customer` indica la fecha de alta, no la fecha de cada compra. La secuencia se interpreta como **orden de incorporación de clientes**, no como historial longitudinal de compras ni como monitoreo de un modelo en producción. `Recency` es una medida relativa a un momento de referencia no incluido en estas columnas y se deja fuera del análisis inferencial principal.

### 1.2. Hipótesis, alcance y reglas de interpretación

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Asegurar que las interpretaciones distingan significancia estadística, magnitud del cambio y alerta secuencial.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** No afirmar que p≥α demuestra igualdad; no interpretar ausencia de alerta ADWIN como evidencia de ausencia de drift.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

- **KS (continuas / numéricas):** H₀: las observaciones proceden de la misma distribución; H₁: difieren. Los empates frecuentes en variables de conteo requieren prudencia interpretativa.
- **Chi-cuadrado (categóricas):** H₀: las proporciones por categoría son homogéneas; H₁: al menos una proporción difiere. Comprobar frecuencias esperadas.
- **PSI:** índice descriptivo, **sin valor p**. Regla orientativa (no universal): `<0.10` cambio bajo, `0.10–<0.25` moderado, `≥0.25` alto. Mantener los cortes definidos en referencia.
- **ADWIN:** señal de cambios secuenciales en el comportamiento de la media de una variable numérica, con potencial retraso en la detección y sensibilidad a fluctuaciones.
- **Múltiples comparaciones:** controlar la tasa de descubrimientos falsos mediante Benjamini–Hochberg por familia de pruebas; conservar también p crudos.
- **Drift artificial:** nunca mezclar resultados simulados con hallazgos observacionales.

## 2. Entorno, dependencias y parámetros

**Responsable principal:** Marco. **Objetivo:** hacer reproducible la ejecución en Google Colab sin trabajo o instalaciones innecesarias.

### 2.1. Instalación mínima e importaciones

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Instalar River solo si no está disponible; reutilizar librerías ya presentes en Colab; registrar versiones y tiempo de ejecución. No reinstalar paquetes grandes indiscriminadamente.

**Entradas y dependencias:** Python 3 y conexión a Internet solo si River no está instalado.

**Criterio de aceptación verificable:** Imports sin fallos en una sesión nueva de Colab y versiones visibles.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
# INSTALACIÓN MÍNIMA: en Colab suelen estar preinstalados pandas, NumPy, SciPy y matplotlib.
import importlib.util, subprocess, sys
if importlib.util.find_spec('river') is None:
    print('River no está instalado. Instalando dependencia ligera para ADWIN...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'river>=0.23,<1'])
else:
    print('River ya disponible; se omite instalación.')

from pathlib import Path
from contextlib import contextmanager
import time
import warnings
import platform
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import chi2_contingency
from tqdm.auto import tqdm
from IPython.display import display
from river import drift
import river

print(f'Python {platform.python_version()} · pandas {pd.__version__} · '
      f'NumPy {np.__version__} · SciPy {__import__("scipy").__version__} · River {river.__version__}')

### 2.2. Configuración, contratos y progreso

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Centralizar las opciones editables, dejar una semilla fija, medir la duración de etapas y declarar explícitamente las listas de variables con columnas reales del CSV.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Un único bloque de configuración, sin números mágicos duplicados; las variables contractuales quedan disponibles para todas las secciones.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
# CONFIGURACIÓN ÚNICA — modificar aquí, no dispersar constantes por las secciones.
CFG = {
    'ruta_csv': None,            # Opcional: ruta local o de Drive; None busca rutas usuales.
    'nombre_csv': 'marketing_campaign.csv',
    'semilla': 42,
    'alpha': 0.05,
    'psi_bins': 10,
    'psi_epsilon': 1e-6,
    'fraccion_referencia': 0.50,
    'adwin_delta': 0.002,
    'adwin_clock': 32,
    'adwin_grace_period': 30,
    'adwin_paso_progreso': 200,
    'adwin_ventana_grafico': 80,
    'mostrar_graficos': True,
    'max_graficos_por_metodo': 4,
}
RNG = np.random.default_rng(CFG['semilla'])
TIEMPOS = {}

NUMERICAS = [
    'Year_Birth', 'Income', 'Kidhome', 'Teenhome',
    'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts',
    'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases',
    'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases',
    'NumWebVisitsMonth',
]
CATEGORICAS = [
    'Education', 'Marital_Status',
    'AcceptedCmp1', 'AcceptedCmp2', 'AcceptedCmp3', 'AcceptedCmp4',
    'AcceptedCmp5', 'Complain', 'Response',
]
ADWIN_COLS = ['Income', 'MntWines', 'MntMeatProducts', 'NumWebPurchases']
COLUMNAS_REQUERIDAS = ['ID', 'Dt_Customer'] + NUMERICAS + CATEGORICAS + [
    'Recency', 'Z_CostContact', 'Z_Revenue',
]
RUTA_RESULTADOS = Path('/content/HM_Data_Drift/resultados') if Path('/content').exists() else Path.cwd() / 'resultados'
RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

@contextmanager
def etapa(nombre):
    print(f'▶ Iniciando: {nombre}', flush=True)
    inicio = time.perf_counter()
    try:
        yield
    finally:
        TIEMPOS[nombre] = time.perf_counter() - inicio
        print(f'✓ Finalizado: {nombre} ({TIEMPOS[nombre]:.2f} s)', flush=True)

print('Parámetros compartidos definidos. Carpeta de resultados:', RUTA_RESULTADOS)

### 2.3. Contrato de integración para agentes

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Congelar nombres, tipos y significado de los artefactos entre secciones; añadir checks de consistencia sin modificar los resultados de otros integrantes.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Todos los participantes emplean `df_clean`, `df_ref`, `df_cur`, `df_seq`, `df_shifted`, `res_ks`, `res_psi`, `res_chi`, `res_adwin`, `res_control`.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

| Artefacto | Responsable que lo crea | Definición de contrato |
|---|---|---|
| `df_raw` | Poma (3) | CSV original, sin modificar |
| `df_clean` | Poma (4) | Datos limpios con `Dt_Customer` como datetime, un registro por ID |
| `df_ref` / `df_cur` | Marco (5) | Cohortes disjuntas por fecha, ordenadas y sin fechas cruzadas |
| `df_seq` | Marco (5) | Concatenación cronológica de referencia + comparación |
| `res_ks` | Vera (7) | DataFrame: variable, n_ref, n_cur, ks, p_valor, p_ajustado, drift_ks, observacion |
| `res_psi` | Vera (8) | DataFrame: variable, psi, intensidad_psi, n_ref, n_cur, bins |
| `res_chi` | Aldana (9) | DataFrame: variable, chi2, p_valor, p_ajustado, drift_chi, supuestos_ok, observacion |
| `res_adwin` / `alertas_adwin` | Brickzon (10) | Resumen por variable y eventos (índice, fecha, valor, anchura) |
| `df_shifted` / `res_control` | Aldana (11) | Comparación alterada **sin tocar** datos originales, resultados etiquetados CONTROLADO |

**Ámbitos de modificación:** Poma puede editar 3/4/6; Vera 7/8; Aldana 9/11; Brickzon 10; Marco 1/2/5/12/13. Para dependencias, **solicitar el cambio al responsable** en lugar de sobrescribir código ajeno.

**Rendimiento:** este CSV es pequeño; usar CPU, NumPy/SciPy vectorizados, un solo `read_csv` y un solo preprocesamiento. Mostrar barra `tqdm` en bucles de variables y en las observaciones de ADWIN, con actualización por lotes para reducir sobrecarga visual. No usar GPU, Dask, multiprocessing ni gráficos de cada variable por defecto.

## 3. Carga y auditoría inicial del dataset

**Responsable principal:** Poma. **Objetivo:** inspeccionar el CSV sin cambiarlo silenciosamente.

### 3.1. Carga eficiente en Colab

**Encargado: Poma.**

**Tarea exacta para el integrante/agente:** Admitir ruta explícita, archivo cargado en /content, Google Drive ya montado y subida manual como último recurso. Validar esquema; detectar TSV usado en los notebooks de referencia.

**Entradas y dependencias:** Sección 2 ejecutada; el CSV debe estar disponible o subirse durante la ejecución.

**Criterio de aceptación verificable:** Carga única `df_raw`, lista completa de columnas, shape y tiempo; error accionable cuando falta CSV.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
def localizar_csv():
    candidatos = []
    if CFG['ruta_csv']:
        candidatos.append(Path(CFG['ruta_csv']))
    candidatos += [
        Path('/content') / CFG['nombre_csv'],
        Path('/content/drive/MyDrive/HM_Data_Drift/datos') / CFG['nombre_csv'],
        Path.cwd() / CFG['nombre_csv'],
        Path('/mnt/data') / CFG['nombre_csv'],
    ]
    for ruta in candidatos:
        if ruta.is_file():
            return ruta
    try:
        from google.colab import files
        print('No se encontró el CSV. Usa el selector para subir marketing_campaign.csv.')
        archivos = files.upload()
        if archivos:
            ruta = Path('/content') / next(iter(archivos.keys()))
            if ruta.exists():
                return ruta
    except ImportError:
        pass
    raise FileNotFoundError(
        'No se encontró marketing_campaign.csv. Sube el CSV a Colab o define CFG["ruta_csv"].'
    )

def cargar_datos(ruta):
    # La muestra original de los notebooks es TSV pese a la extensión .csv.
    df = pd.read_csv(ruta, sep='\t', encoding='utf-8-sig', low_memory=False)
    if df.shape[1] == 1:
        print('El archivo no parece TSV: probando detección de separador una sola vez.')
        df = pd.read_csv(ruta, sep=None, engine='python', encoding='utf-8-sig')
    faltantes = sorted(set(COLUMNAS_REQUERIDAS) - set(df.columns))
    if faltantes:
        raise ValueError(f'CSV incompatible: faltan columnas {faltantes}. Columnas reales: {list(df.columns)}')
    return df

with etapa('Carga del CSV'):
    RUTA_CSV = localizar_csv()
    df_raw = cargar_datos(RUTA_CSV)
print(f'Archivo: {RUTA_CSV} · Filas: {len(df_raw):,} · Columnas: {df_raw.shape[1]}')
display(df_raw.head(5))

### 3.2. Perfil y calidad original

**Encargado: Poma.**

**Tarea exacta para el integrante/agente:** Informar valores nulos, duplicados e inconsistencias visibles, sin imputar ni borrar registros en este paso.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Tabla de calidad original visible y explicación de anomalías/limitaciones; sin alterar `df_raw`.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
with etapa('Perfil del dataset'):
    calidad = pd.DataFrame({
        'columna': df_raw.columns,
        'tipo_original': df_raw.dtypes.astype(str).to_numpy(),
        'nulos': df_raw.isna().sum().to_numpy(),
        'n_unicos': df_raw.nunique(dropna=False).to_numpy(),
    })
    calidad['pct_nulos'] = (100 * calidad['nulos'] / len(df_raw)).round(2)
    print(f'Filas totalmente duplicadas: {int(df_raw.duplicated().sum())}; '
          f'IDs duplicados: {int(df_raw["ID"].duplicated().sum())}')
    display(calidad)
    display(df_raw[NUMERICAS].describe().T.round(2))

## 4. Preprocesamiento y selección de variables

**Responsable principal:** Poma. **Objetivo:** generar un dataset seguro, tipado y sin alteraciones arbitrarias de distribuciones.

### 4.1. Normalización conservadora y reglas de calidad

**Encargado: Poma.**

**Tarea exacta para el integrante/agente:** Convertir fechas con formato día-mes-año, números con `to_numeric`, categorías normalizadas. Eliminar solo duplicados idénticos; ante IDs contradictorios fallar; descartar fechas inválidas (registrándolo). No imputar ingresos ni recortar atípicos: cada prueba usa casos completos de su variable.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Función `preparar_dataset` determinista, reporte de cambios y ninguna alteración del CSV original.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
def preparar_dataset(raw):
    df = raw.copy()
    reporte = {'filas_iniciales': len(df)}
    duplicadas = int(df.duplicated().sum())
    df = df.drop_duplicates().copy()
    reporte['duplicados_exactos_eliminados'] = duplicadas
    df['ID'] = pd.to_numeric(df['ID'], errors='raise').astype('int64')
    duplicados_id = int(df['ID'].duplicated().sum())
    if duplicados_id:
        raise ValueError(f'Hay {duplicados_id} IDs repetidos con contenidos distintos; resolver manualmente.')
    df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'], format='%d-%m-%Y', errors='coerce')
    fechas_invalidas = int(df['Dt_Customer'].isna().sum())
    reporte['fechas_invalidas_eliminadas'] = fechas_invalidas
    df = df.loc[df['Dt_Customer'].notna()].copy()
    for col in NUMERICAS + ['Recency', 'Z_CostContact', 'Z_Revenue']:
        df[col] = pd.to_numeric(df[col], errors='coerce')
    for col in CATEGORICAS:
        # Se preserva la ausencia como una categoría explícita y auditable.
        df[col] = df[col].astype('string').str.strip().fillna('__FALTANTE__')
        df.loc[df[col].eq(''), col] = '__FALTANTE__'
    if len(df) < 40:
        raise ValueError('Quedan menos de 40 filas válidas; no es posible la comparación propuesta.')
    reporte['filas_finales'] = len(df)
    return df.reset_index(drop=True), pd.Series(reporte, name='valor')

with etapa('Preparación conservadora'):
    df_clean, reporte_limpieza = preparar_dataset(df_raw)
display(reporte_limpieza.to_frame())
print('Nulos por variable numérica (se conservarán hasta cada prueba):')
display(df_clean[NUMERICAS].isna().sum().rename('nulos').to_frame().query('nulos > 0'))

### 4.2. Variables excluidas y auditoría de rangos

**Encargado: Poma.**

**Tarea exacta para el integrante/agente:** Justificar exclusión de ID, fecha como feature, constantes Z_* y Recency del conjunto principal; describir valores extremos sin decidir borrados automáticos.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Queda una tabla de anomalías/rangos y explicaciones; ningún outlier se elimina por conveniencia.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
with etapa('Auditoría de extremos'):
    extremos = df_clean[NUMERICAS + ['Recency']].agg(['min', 'median', 'max']).T
    extremos['nulos'] = df_clean[extremos.index].isna().sum()
    display(extremos.round(2))
    print('Valores únicos de constantes auxiliares:')
    display(df_clean[['Z_CostContact', 'Z_Revenue']].nunique(dropna=False).rename('n_unicos').to_frame())
    print('Nota: Year_Birth y valores monetarios anómalos se reportan, NO se recortan automáticamente.')

## 5. División temporal y contrato de cohortes

**Responsable principal:** Marco. **Objetivo:** producir conjuntos disjuntos temporalmente y reproducibles.

### 5.1. Corte cronológico sin dividir una fecha

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Ordenar estable por fecha e ID. Elegir el corte que mejor aproxima la fracción objetivo sin separar filas de una misma fecha. Garantizar al menos dos fechas y muestras útiles.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Las fechas de `df_ref` preceden estrictamente a las de `df_cur`; no hay IDs compartidos; `df_seq` conserva el orden.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
def crear_cohortes(df, fraccion=0.50):
    orden = df.sort_values(['Dt_Customer', 'ID'], kind='mergesort').reset_index(drop=True)
    por_fecha = orden.groupby('Dt_Customer', sort=True).size()
    if len(por_fecha) < 2:
        raise ValueError('Se requieren por lo menos dos fechas de incorporación diferentes.')
    acumulado = por_fecha.cumsum().iloc[:-1]
    objetivo = len(orden) * fraccion
    posicion = int(np.argmin(np.abs(acumulado.to_numpy() - objetivo)))
    n_ref = int(acumulado.iloc[posicion])
    ref, cur = orden.iloc[:n_ref].copy(), orden.iloc[n_ref:].copy()
    if len(ref) < 20 or len(cur) < 20:
        raise ValueError('Partición cronológica demasiado pequeña; revisar fechas de incorporación.')
    assert ref['Dt_Customer'].max() < cur['Dt_Customer'].min()
    assert set(ref['ID']).isdisjoint(set(cur['ID']))
    return ref.reset_index(drop=True), cur.reset_index(drop=True), orden

with etapa('Construcción de cohortes'):
    df_ref, df_cur, df_seq = crear_cohortes(df_clean, CFG['fraccion_referencia'])
print(f'REFERENCIA: n={len(df_ref):,} | {df_ref.Dt_Customer.min().date()} a {df_ref.Dt_Customer.max().date()}')
print(f'COMPARACIÓN: n={len(df_cur):,} | {df_cur.Dt_Customer.min().date()} a {df_cur.Dt_Customer.max().date()}')
print(f'Primer registro comparación: ID={df_cur.iloc[0].ID}, fecha={df_cur.iloc[0].Dt_Customer.date()}')

### 5.2. Controles de reproducibilidad y sesgos

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Comprobar que las cohortes no se solapan, que todas las filas limpias entran una sola vez y que existe suficiente observación por variable.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Todas las aserciones pasan; se registra que no se interpreta esto como evolución longitudinal.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
assert len(df_ref) + len(df_cur) == len(df_clean)
assert df_seq['Dt_Customer'].is_monotonic_increasing
assert df_seq['ID'].is_unique
print('PASS: cohortes disjuntas por fecha e ID; orden estable; sin pérdida de filas limpias.')
conteo_util = pd.DataFrame({
    'variable': NUMERICAS,
    'n_ref': [int(df_ref[c].notna().sum()) for c in NUMERICAS],
    'n_cur': [int(df_cur[c].notna().sum()) for c in NUMERICAS],
})
display(conteo_util)

## 6. Análisis exploratorio comparativo

**Responsable principal:** Poma. **Objetivo:** mostrar diferencias descriptivas antes de aplicar pruebas.

### 6.1. Resumen por cohortes

**Encargado: Poma.**

**Tarea exacta para el integrante/agente:** Comparar mediana, media, dispersión y porcentaje de faltantes en cuatro variables representativas; evitar gráficas de todas las columnas.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Tabla interpretada; los nulos de Income quedan transparentes y no se imputan.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
with etapa('Resumen exploratorio'):
    cols_eda = ['Income','MntWines','MntMeatProducts','NumWebPurchases']
    resumen_eda = pd.concat([
        df_ref[cols_eda].agg(['count','mean','median','std']).T.assign(cohorte='referencia'),
        df_cur[cols_eda].agg(['count','mean','median','std']).T.assign(cohorte='comparacion'),
    ]).rename_axis('variable').reset_index()
    display(resumen_eda.round(2))

### 6.2. Distribuciones visuales con límite de gráficos

**Encargado: Poma.**

**Tarea exacta para el integrante/agente:** Trazar solo las variables ilustrativas y dos categorías; compartir ejes y leyendas; no generar decenas de gráficos automáticamente.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Gráficos comparables, legibles y sin confundir densidades con conteos.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
if CFG['mostrar_graficos']:
    with etapa('Gráficos EDA'):
        variables = ['Income', 'MntWines', 'NumWebPurchases'][:CFG['max_graficos_por_metodo']]
        for col in tqdm(variables, desc='EDA numérica', unit='gráfico'):
            a = df_ref[col].dropna().to_numpy(dtype=float)
            b = df_cur[col].dropna().to_numpy(dtype=float)
            fig, ax = plt.subplots(figsize=(8, 3.2))
            ax.hist(a, bins=25, alpha=.45, density=True, label='Referencia')
            ax.hist(b, bins=25, alpha=.45, density=True, label='Comparación')
            ax.set(title=f'Distribución comparativa: {col}', xlabel=col, ylabel='Densidad')
            ax.legend(); fig.tight_layout(); plt.show(); plt.close(fig)
        for col in ['Education', 'Marital_Status']:
            tabla = pd.DataFrame({
                'Referencia': df_ref[col].value_counts(normalize=True),
                'Comparación': df_cur[col].value_counts(normalize=True),
            }).fillna(0).sort_index()
            ax = tabla.plot(kind='bar', figsize=(8, 3), title=f'Proporciones: {col}')
            ax.set(ylabel='Proporción', xlabel=col)
            plt.tight_layout(); plt.show(); plt.close(ax.figure)

## 7. Kolmogorov-Smirnov para variables numéricas

**Responsable principal:** Vera. **Objetivo:** comparar distribuciones con control de comparaciones múltiples.

### 7.1. Función KS, justificación y resultados

**Encargado: Vera.**

**Tarea exacta para el integrante/agente:** Aplicar `scipy.stats.ks_2samp` sobre arrays limpios, informar n, estadístico y p. Añadir ajuste BH y advertir empates en variables discretas.

**Entradas y dependencias:** `df_ref`, `df_cur`, `NUMERICAS` creados en secciones 4–5.

**Criterio de aceptación verificable:** `res_ks` con columnas contractuales, p ajustado válido, resumen por variable y progreso del bucle.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Guía teórica:** H₀ indica mismas distribuciones; el estadístico D es la separación máxima entre funciones de distribución empírica. La significancia se evalúa sobre `p_ajustado < alpha`, y un no-rechazo **no equivale** a distribuciones idénticas. En columnas de conteo, interpretar con cautela por los empates.

In [ ]:
def ajustar_bh(pvalores):
    p = np.asarray(pvalores, dtype=float)
    salida = np.full(len(p), np.nan)
    validos = np.flatnonzero(np.isfinite(p))
    if not len(validos):
        return salida
    orden = validos[np.argsort(p[validos])]
    m = len(orden)
    ajuste = np.minimum.accumulate((p[orden] * m / np.arange(1, m+1))[::-1])[::-1]
    salida[orden] = np.clip(ajuste, 0, 1)
    return salida

def calcular_ks(ref, cur, columnas, alpha=0.05, progreso=True):
    filas = []
    iterador = tqdm(columnas, desc='KS', unit='variable') if progreso else columnas
    for col in iterador:
        a = ref[col].dropna().to_numpy(dtype=float)
        b = cur[col].dropna().to_numpy(dtype=float)
        fila = {'variable':col, 'n_ref':len(a), 'n_cur':len(b),
                'ks':np.nan, 'p_valor':np.nan, 'observacion':''}
        if min(len(a), len(b)) < 20:
            fila['observacion'] = 'Muestra insuficiente (<20 por cohorte)'
        else:
            resultado = stats.ks_2samp(a, b, method='auto')
            fila.update(ks=float(resultado.statistic), p_valor=float(resultado.pvalue))
            if min(len(np.unique(a)), len(np.unique(b))) < 10:
                fila['observacion'] = 'Numerosos empates; interpretación KS aproximada'
        filas.append(fila)
    out = pd.DataFrame(filas)
    out['p_ajustado'] = ajustar_bh(out['p_valor'])
    out['drift_ks'] = out['p_ajustado'].lt(alpha) & out['p_ajustado'].notna()
    return out[['variable','n_ref','n_cur','ks','p_valor','p_ajustado','drift_ks','observacion']]

with etapa('Prueba Kolmogorov-Smirnov'):
    res_ks = calcular_ks(df_ref, df_cur, NUMERICAS, CFG['alpha'])
display(res_ks.sort_values(['drift_ks','p_ajustado'],ascending=[False,True]).round(5))

### 7.2. Lectura de la evidencia KS

**Encargado: Vera.**

**Tarea exacta para el integrante/agente:** Seleccionar hasta tres variables significativas (o menores p) y describir sentido y relevancia junto con las advertencias por empates.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Redactar un párrafo de interpretación **después** de ejecutar el CSV, sin usar conclusiones de resultados artificiales.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Espacio de interpretación de Vera (pendiente de datos reales):** registrar variables con menor `p_ajustado`, estadístico D, tamaños de muestra, cautelas de datos discretos y diferencia entre significancia y magnitud.

## 8. Population Stability Index (PSI)

**Responsable principal:** Vera. **Objetivo:** cuantificar diferencias usando bins fijados sobre la referencia.

### 8.1. PSI robusto y reproducible

**Encargado: Vera.**

**Tarea exacta para el integrante/agente:** Calcular cuantiles solo con referencia, colapsar cortes repetidos, incluir extremos infinitos, proporciones suavizadas y clasificar magnitud con umbrales explícitos.

**Entradas y dependencias:** Las mismas cohortes y columnas del KS.

**Criterio de aceptación verificable:** `res_psi` completo, resultados finitos incluso con columnas de baja cardinalidad; barras de progreso.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Definición:** para cada bin `i`, `PSI = Σ (p_cur,i − p_ref,i) · ln(p_cur,i / p_ref,i)`. Se usan **bins calculados exclusivamente sobre referencia**, compartidos con comparación. Un `epsilon` evita logaritmos de cero; los porcentajes se renormalizan tras suavizar. PSI no ofrece significancia estadística y los umbrales son heurísticos.

In [ ]:
def psi_variable(referencia, comparacion, bins=10, epsilon=1e-6):
    a = np.asarray(referencia, dtype=float)
    b = np.asarray(comparacion, dtype=float)
    a, b = a[np.isfinite(a)], b[np.isfinite(b)]
    if min(len(a), len(b)) < 20:
        return np.nan, 0
    # Los cortes se obtienen solo de referencia; un mismo bin se utiliza en ambas.
    cuantiles = np.quantile(a, np.linspace(0, 1, bins + 1)[1:-1])
    cortes = np.unique(cuantiles)
    limites = np.concatenate(([-np.inf], cortes, [np.inf]))
    cuenta_a, _ = np.histogram(a, bins=limites)
    cuenta_b, _ = np.histogram(b, bins=limites)
    p_a = cuenta_a / len(a)
    p_b = cuenta_b / len(b)
    p_a = np.maximum(p_a, epsilon); p_a = p_a / p_a.sum()
    p_b = np.maximum(p_b, epsilon); p_b = p_b / p_b.sum()
    valor = float(np.sum((p_b - p_a) * np.log(p_b / p_a)))
    return valor, len(limites)-1

def categoria_psi(v):
    if not np.isfinite(v): return 'No evaluable'
    if v < .10: return 'Bajo'
    if v < .25: return 'Moderado'
    return 'Alto'

def calcular_psi(ref, cur, columnas, bins=10, epsilon=1e-6, progreso=True):
    filas=[]
    iterable = tqdm(columnas, desc='PSI', unit='variable') if progreso else columnas
    for col in iterable:
        psi, n_bins = psi_variable(ref[col].to_numpy(), cur[col].to_numpy(), bins, epsilon)
        filas.append({'variable':col, 'psi':psi, 'intensidad_psi':categoria_psi(psi),
                      'n_ref':int(ref[col].notna().sum()), 'n_cur':int(cur[col].notna().sum()),
                      'bins':n_bins})
    return pd.DataFrame(filas)

with etapa('Population Stability Index'):
    res_psi = calcular_psi(df_ref, df_cur, NUMERICAS, CFG['psi_bins'], CFG['psi_epsilon'])
display(res_psi.sort_values('psi', ascending=False).round(5))

### 8.2. Validación y lectura de PSI

**Encargado: Vera.**

**Tarea exacta para el integrante/agente:** Revisar que los PSI sean no negativos salvo redondeos, explicar el número efectivo de bins y contrastar PSI con KS sin equiparar métricas.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Pruebas de sanidad pasan e interpretación no atribuye valor p al PSI.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
# Autopruebas pequeñas independientes del CSV real.
valor_igual, _ = psi_variable(np.arange(100, dtype=float), np.arange(100, dtype=float))
assert abs(valor_igual) < 1e-12
assert res_psi['psi'].dropna().ge(-1e-10).all()
print('PASS: PSI idéntico ≈ 0 y resultados no negativos.')

**Espacio de interpretación de Vera:** explicar las variables con PSI alto/moderado y la diferencia frente al rechazo/no-rechazo de KS. Completar tras ejecutar el CSV.

## 9. Chi-cuadrado para variables categóricas

**Responsable principal:** Aldana. **Objetivo:** comparar proporciones con chequeo de supuestos.

### 9.1. Prueba de homogeneidad y frecuencias esperadas

**Encargado: Aldana.**

**Tarea exacta para el integrante/agente:** Construir tabla 2×k con las categorías conjuntas; descartar columnas de suma nula, comprobar esperados; ajustar p BH; diferenciar “no concluyente” de “sin drift”.

**Entradas y dependencias:** `df_ref`, `df_cur`, `CATEGORICAS`.

**Criterio de aceptación verificable:** `res_chi` con supuestos_ok y evidencias; progreso; p solo se interpreta formalmente si esperados son aceptables.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
def calcular_chi(ref, cur, columnas, alpha=0.05, progreso=True):
    filas=[]
    iterable = tqdm(columnas, desc='Chi²', unit='variable') if progreso else columnas
    for col in iterable:
        a=ref[col].astype('string').fillna('__FALTANTE__').value_counts()
        b=cur[col].astype('string').fillna('__FALTANTE__').value_counts()
        categorias=a.index.union(b.index)
        tabla=np.vstack([a.reindex(categorias,fill_value=0).to_numpy(),
                         b.reindex(categorias,fill_value=0).to_numpy()])
        tabla=tabla[:,tabla.sum(axis=0)>0]
        fila={'variable':col, 'chi2':np.nan, 'p_valor':np.nan,
              'supuestos_ok':False, 'observacion':''}
        if tabla.shape[1] < 2:
            fila['observacion']='Una sola categoría: contraste no definido'
        else:
            prueba=chi2_contingency(tabla, correction=False)
            esperado=np.asarray(prueba.expected_freq)
            fila['chi2']=float(prueba.statistic)
            fila['supuestos_ok']=bool((esperado >= 5).all())
            if fila['supuestos_ok']:
                fila['p_valor']=float(prueba.pvalue)
                fila['observacion']='Supuestos mínimos cumplidos'
            else:
                fila['observacion']='Frecuencias esperadas <5: no concluyente con chi² asintótico'
        filas.append(fila)
    out=pd.DataFrame(filas)
    out['p_ajustado']=ajustar_bh(out['p_valor'])
    out['drift_chi']=out['supuestos_ok'] & out['p_ajustado'].lt(alpha)
    return out[['variable','chi2','p_valor','p_ajustado','drift_chi','supuestos_ok','observacion']]

with etapa('Chi-cuadrado categórico'):
    res_chi=calcular_chi(df_ref, df_cur, CATEGORICAS, CFG['alpha'])
display(res_chi.sort_values(['drift_chi','p_ajustado'],ascending=[False,True]).round(5))

### 9.2. Interpretación de categorías y advertencias

**Encargado: Aldana.**

**Tarea exacta para el integrante/agente:** Señalar categorías con mayores diferencias porcentuales y reportar variables que no cumplen frecuencias esperadas; no ocultar los no concluyentes.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Dos ejemplos interpretados con tabla de proporciones; no se publica “drift=no” cuando no se puede evaluar.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
for col in ['Education','Marital_Status']:
    resumen=pd.DataFrame({
        'p_ref':df_ref[col].value_counts(normalize=True),
        'p_cur':df_cur[col].value_counts(normalize=True),
    }).fillna(0)
    resumen['delta_pp']=(resumen['p_cur']-resumen['p_ref'])*100
    print(f'Cambios de proporción: {col} (puntos porcentuales)')
    display(resumen.sort_values('delta_pp',key=abs,ascending=False).round(3))

**Espacio de interpretación de Aldana:** describir cambios en categorías dominantes, variables con supuestos inválidos y alcance de la inferencia.

## 10. Detección secuencial mediante ADWIN

**Responsable principal:** Brickzon. **Objetivo:** encontrar alertas de cambio en la media a lo largo del orden de incorporación.

### 10.1. Procesamiento en streaming y barra de progreso

**Encargado: Brickzon.**

**Tarea exacta para el integrante/agente:** Crear un detector fresco por variable; escalar usando parámetros **solo de referencia** para estabilidad numérica; saltar nulos, procesar elemento por elemento con `update`; registrar alertas con índice/fecha, anchura y retardo. Actualizar tqdm cada N observaciones, no cada evento de render.

**Entradas y dependencias:** API River `drift.ADWIN(delta=..., clock=..., grace_period=...)`; fuente: https://riverml.xyz/latest/api/drift/ADWIN/.

**Criterio de aceptación verificable:** `ejecutar_adwin(df_seq, df_ref, columnas, ...)` retorna resumen y eventos sin modificar entradas, mostrando barra de progreso y tiempos.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Detalle metodológico:** ADWIN utiliza observaciones secuenciales; aquí cada evento es un cliente ordenado por su fecha de incorporación. La transformación `(x − mediana_ref) / escala_ref` se fija **solo con la cohorte de referencia**, para que la misma magnitud represente la misma escala en todo el flujo. No hay entrenamiento supervisado. Fecha de alerta ≠ fecha exacta del cambio real: ADWIN puede detectar con retraso. No interpretar cambios en `MntWines` como evolución histórica de precios o ventas.

In [ ]:
def ejecutar_adwin(secuencia, referencia, columnas, delta=0.002, clock=32,
                  grace_period=30, paso=200, progreso=True):
    eventos=[]; resumen=[]
    total = len(secuencia)*len(columnas)
    barra = tqdm(total=total, desc='ADWIN: observaciones', unit='obs',
                 mininterval=0.4, disable=not progreso)
    try:
        for col in columnas:
            base=referencia[col].dropna().to_numpy(dtype=float)
            if len(base) < 20:
                resumen.append({'variable':col,'alertas':0,'procesadas':0,
                                'omitidas':len(secuencia), 'estado':'Sin referencia suficiente'})
                barra.update(len(secuencia))
                continue
            mediana=float(np.median(base))
            iqr=float(np.subtract(*np.percentile(base,[75,25])))
            escala=iqr if iqr > 1e-12 else float(np.std(base))
            escala=escala if escala > 1e-12 else 1.0
            detector=drift.ADWIN(delta=delta,clock=clock,grace_period=grace_period)
            serie=secuencia[col].to_numpy(dtype=float)
            fechas=secuencia['Dt_Customer'].to_numpy()
            ids=secuencia['ID'].to_numpy()
            n_validas=0; n_invalidas=0; n_alertas=0; pendientes=0
            barra.set_postfix_str(f'variable={col}', refresh=False)
            for i,valor in enumerate(serie):
                pendientes += 1
                if np.isfinite(valor):
                    n_validas += 1
                    detector.update(float((valor-mediana)/escala))
                    if detector.drift_detected:
                        n_alertas += 1
                        eventos.append({'variable':col, 'indice_0based':i,
                                        'numero_observacion':i+1, 'ID':int(ids[i]),
                                        'fecha':pd.Timestamp(fechas[i]),
                                        'valor_original':float(valor),
                                        'ancho_ventana':int(detector.width)})
                else:
                    n_invalidas += 1
                if pendientes >= paso:
                    barra.update(pendientes)
                    pendientes=0
            if pendientes: barra.update(pendientes)
            resumen.append({'variable':col,'alertas':n_alertas,'procesadas':n_validas,
                            'omitidas':n_invalidas,'estado':'Evaluado'})
    finally:
        barra.close()
    return (pd.DataFrame(resumen,columns=['variable','alertas','procesadas','omitidas','estado']),
            pd.DataFrame(eventos,columns=['variable','indice_0based','numero_observacion',
                        'ID','fecha','valor_original','ancho_ventana']))

with etapa('ADWIN, datos observados'):
    res_adwin, alertas_adwin=ejecutar_adwin(
        df_seq, df_ref, ADWIN_COLS,
        delta=CFG['adwin_delta'], clock=CFG['adwin_clock'],
        grace_period=CFG['adwin_grace_period'], paso=CFG['adwin_paso_progreso'])
display(res_adwin)
if len(alertas_adwin): display(alertas_adwin)
else: print('ADWIN no emitió alertas con esta configuración; no equivale a ausencia probada de drift.')

### 10.2. Gráficos de alertas y desplazamiento de detección

**Encargado: Brickzon.**

**Tarea exacta para el integrante/agente:** Graficar media móvil descriptiva de máximo cuatro variables y superponer alertas reales; mostrar línea del corte de cohortes para facilitar interpretación. Limitar cantidad de renderizados.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Gráficos con punto de corte y alertas identificables, incluso si no hay eventos.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
def graficar_adwin(secuencia, eventos, columnas, corte, ventana=80):
    for col in tqdm(columnas, desc='Gráficos ADWIN', unit='gráfico'):
        y=secuencia[col].astype(float).rolling(ventana,min_periods=max(5,ventana//4)).mean()
        ev=eventos.loc[eventos['variable'].eq(col),'indice_0based'] if len(eventos) else []
        fig,ax=plt.subplots(figsize=(10,3.2))
        ax.plot(np.arange(len(y)), y.to_numpy(), label=f'Media móvil ({ventana})')
        ax.axvline(corte,linestyle='--',label='Inicio cohorte comparación')
        for j,pos in enumerate(ev):
            ax.axvline(pos,color='tab:red',alpha=.65,linewidth=1,
                       label='Alerta ADWIN' if j==0 else None)
        ax.set(title=f'ADWIN: {col}',xlabel='Orden de incorporación (cliente)',ylabel=col)
        ax.legend(loc='best'); fig.tight_layout();plt.show();plt.close(fig)

if CFG['mostrar_graficos']:
    with etapa('Visualización de alertas ADWIN'):
        graficar_adwin(df_seq, alertas_adwin,
                      ADWIN_COLS[:CFG['max_graficos_por_metodo']],len(df_ref),
                      CFG['adwin_ventana_grafico'])

### 10.3. Interpretación cautelosa de detecciones

**Encargado: Brickzon.**

**Tarea exacta para el integrante/agente:** Explicar para cada alerta la fecha de incorporación y el índice; distinguir cambio en orden de cohortes de cambio temporal de compras.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Texto relaciona cada evento con la serie analizada y menciona potencial falso positivo, retraso y ausencia de fechas de transacción.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Espacio de interpretación de Brickzon:** registrar alertas observadas, variables afectadas y su relación con el corte de cohortes. Si no hay alertas, indicar exactamente eso, sin elevar la conclusión a una prueba de ausencia de drift.

## 11. Experimentos de drift artificial controlado

**Responsable principal:** Aldana. **Objetivo:** validar capacidad de detección usando alteraciones explícitas, reproducibles y separadas.

### 11.1. Inyección artificial no destructiva

**Encargado: Aldana.**

**Tarea exacta para el integrante/agente:** Construir una copia del conjunto comparación y alterar ingresos (+25%), gastos en vinos (+60%) y convertir el 35% de no casados a Married con RNG fija; registrar cambios efectivos, nunca cambiar `df_cur`.

**Entradas y dependencias:** Datos preparados en sección 5.

**Criterio de aceptación verificable:** `df_shifted` tiene mismos IDs/fechas/columnas que comparación, `df_cur` permanece intacto y se registran tasas efectivas.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
def inyectar_drift(cur, semilla=42):
    shifted=cur.copy(deep=True)
    generador=np.random.default_rng(semilla)
    shifted['Income']=shifted['Income']*1.25
    shifted['MntWines']=shifted['MntWines']*1.60
    mascara=shifted['Marital_Status'].ne('Married').to_numpy(dtype=bool)
    seleccion=mascara & (generador.random(len(shifted)) < 0.35)
    shifted.loc[seleccion,'Marital_Status']='Married'
    reporte={
      'filas_comparacion':len(shifted),
      'income_modificados':int(cur['Income'].notna().sum()),
      'vinos_modificados_no_cero':int((cur['MntWines'].fillna(0)!=0).sum()),
      'estado_civil_modificados':int(seleccion.sum()),
    }
    return shifted, pd.Series(reporte,name='valor')

with etapa('Drift artificial'):
    df_shifted, reporte_inyeccion=inyectar_drift(df_cur,CFG['semilla'])
assert len(df_shifted)==len(df_cur)
assert df_shifted['ID'].equals(df_cur['ID'])
assert df_shifted['Dt_Customer'].equals(df_cur['Dt_Customer'])
assert df_shifted['Marital_Status'].ne(df_cur['Marital_Status']).sum()==int(reporte_inyeccion['estado_civil_modificados'])
display(reporte_inyeccion.to_frame())
print('IMPORTANTE: se generó una COPIA alterada. Los datos observados permanecen intactos.')

### 11.2. Evaluación cruzada con KS, PSI, chi-cuadrado y ADWIN

**Encargado: Aldana.**

**Tarea exacta para el integrante/agente:** Reutilizar funciones de Vera y Brickzon sin reimplementarlas. Comparar referencia con comparación artificial para métodos batch y procesar secuencia ref + cur modificada para ADWIN.

**Entradas y dependencias:** `calcular_ks`, `calcular_psi`, `calcular_chi`, `ejecutar_adwin` deben existir sin cambios de firma.

**Criterio de aceptación verificable:** `res_control` distingue escenario CONTROLADO, indicadores comparables y alertas ADWIN posteriores al punto de intervención.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
with etapa('Evaluación de experimento controlado'):
    control_ks=calcular_ks(df_ref,df_shifted,NUMERICAS,CFG['alpha'])
    control_psi=calcular_psi(df_ref,df_shifted,NUMERICAS,CFG['psi_bins'],CFG['psi_epsilon'])
    control_chi=calcular_chi(df_ref,df_shifted,CATEGORICAS,CFG['alpha'])
    df_seq_control=pd.concat([df_ref,df_shifted],ignore_index=True)
    control_adwin, alertas_control=ejecutar_adwin(
        df_seq_control,df_ref,ADWIN_COLS,
        delta=CFG['adwin_delta'],clock=CFG['adwin_clock'],
        grace_period=CFG['adwin_grace_period'],paso=CFG['adwin_paso_progreso'])
    alerta_post=(alertas_control.loc[alertas_control['indice_0based'].ge(len(df_ref))]
                  if len(alertas_control) else alertas_control)

res_control=(control_ks[['variable','drift_ks','p_ajustado']]
    .merge(control_psi[['variable','psi','intensidad_psi']],on='variable',how='outer')
    .merge(control_adwin[['variable','alertas']],on='variable',how='left')
    .rename(columns={'alertas':'alertas_adwin_total'}))
res_control['escenario']='CONTROLADO'
res_control['alertas_adwin_posteriores']=res_control['variable'].map(
    alerta_post['variable'].value_counts() if len(alerta_post) else {}).fillna(0).astype(int)
print('CONTROLADO: variables numéricas modificadas directamente')
display(res_control.loc[res_control['variable'].isin(['Income','MntWines'])].round(4))
print('CONTROLADO: categorías de estado civil')
display(control_chi.loc[control_chi['variable'].eq('Marital_Status')])
print('Nota: no debe exigirse que ADWIN detecte exactamente el índice de la intervención.')

### 11.3. Evidencia experimental y límites

**Encargado: Aldana.**

**Tarea exacta para el integrante/agente:** Contrastar detecciones observadas frente al cambio conocido y calcular el primer retraso de ADWIN si hay alerta después del punto de inyección; no afirmar éxito de un algoritmo sin medirlo.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Reporte de cambio inducido, si hubo alerta posterior y su retraso; interpretación separada del escenario real.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
corte_control=len(df_ref)
if len(alerta_post):
    primera=(alerta_post.sort_values('indice_0based').groupby('variable',as_index=False)
             .first()[['variable','indice_0based','fecha']])
    primera['retraso_observaciones']=primera['indice_0based']-corte_control+1
    print('Primeras alertas POSTERIORES al inicio de la comparación alterada:')
    display(primera)
else:
    print('No se detectaron alertas posteriores al punto de intervención bajo estos parámetros.')
print('El experimento introduce cambios de forma deliberada y no permite inferir causas en los datos originales.')

**Espacio de interpretación de Aldana:** especificar magnitud del cambio inducido, pruebas que lo detectaron, retraso de ADWIN y discrepancias entre detectores.

## 12. Consolidación, validaciones y resultados exportables

**Responsable principal:** Marco. **Objetivo:** integrar evidencia sin confundir métodos ni escenarios.

### 12.1. Tabla comparativa única

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Unir resultados por variable de forma trazable; no rellenar métodos no aplicables con cero; registrar variable evaluable/no evaluable y escala de resultados.

**Entradas y dependencias:** Secciones 7–11 concluidas.

**Criterio de aceptación verificable:** `resumen_final` incluye columnas numéricas y categóricas, con etiquetas de método y sin presentar “chi² falso” cuando supuestos fallan.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
with etapa('Consolidación de métodos'):
    tabla_numerica=(res_ks.merge(res_psi[['variable','psi','intensidad_psi']],on='variable',how='outer')
        .merge(res_adwin[['variable','alertas']],on='variable',how='left')
        .rename(columns={'alertas':'alertas_adwin'}))
    tabla_numerica['tipo']='numérica'
    tabla_categorica=res_chi.copy()
    tabla_categorica['tipo']='categórica'
    resumen_final=pd.concat([tabla_numerica,tabla_categorica],ignore_index=True,sort=False)
    resumen_final=resumen_final.sort_values(['tipo','variable']).reset_index(drop=True)
    display(resumen_final.round(4))

### 12.2. Pruebas automáticas y perfil de tiempos

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Validar p, estabilidad de partición, columnas esperadas, escenario sin fuga de datos y capacidad de reproducción; mostrar las etapas más costosas.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Checks críticos pasan; tabla de tiempos identifica cuellos de botella sin inventarlos.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
with etapa('Auditoría automática'):
    assert len(df_ref)+len(df_cur)==len(df_clean)
    assert set(df_ref.ID).isdisjoint(set(df_cur.ID))
    assert df_ref.Dt_Customer.max() < df_cur.Dt_Customer.min()
    assert set(res_ks.variable)==set(NUMERICAS)
    assert set(res_psi.variable)==set(NUMERICAS)
    assert set(res_chi.variable)==set(CATEGORICAS)
    assert set(res_adwin.variable)==set(ADWIN_COLS)
    for tabla in [res_ks,res_chi]:
        assert tabla['p_valor'].dropna().between(0,1).all()
        assert tabla['p_ajustado'].dropna().between(0,1).all()
    assert res_psi['psi'].dropna().ge(-1e-9).all()
    assert df_shifted['ID'].equals(df_cur['ID'])
    print('PASS: contratos, valores p, PSI, cohortes y copia experimental.')

tiempos=pd.Series(TIEMPOS,name='segundos').sort_values(ascending=False).round(3)
display(tiempos.to_frame())

### 12.3. Exportación compacta y trazabilidad

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Exportar tablas en CSV, fichero de configuración/manifest con fechas, n, librerías y semilla; crear ZIP para descarga desde Colab. No guardar PII fuera de IDs en tablas de eventos salvo necesidad.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Archivos guardados en carpeta de resultados y ZIP indicado. No se exporta copia de todo el dataset ni cambios artificiales como observaciones reales.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

In [ ]:
import json, shutil
with etapa('Exportación de resultados'):
    entregables={
      'resumen_final.csv':resumen_final,
      'ks_observado.csv':res_ks,
      'psi_observado.csv':res_psi,
      'chi_observado.csv':res_chi,
      'adwin_resumen_observado.csv':res_adwin,
      'adwin_alertas_observado.csv':alertas_adwin,
      'experimento_controlado.csv':res_control,
      'chi_controlado.csv':control_chi,
    }
    for nombre,tabla in tqdm(entregables.items(),desc='Exportación',unit='archivo'):
        tabla.to_csv(RUTA_RESULTADOS/nombre,index=False,encoding='utf-8-sig')
    manifiesto={
      'fuente':str(RUTA_CSV), 'filas_originales':len(df_raw),
      'filas_limpias':len(df_clean), 'referencia_n':len(df_ref),
      'comparacion_n':len(df_cur),
      'referencia_fecha_fin':str(df_ref.Dt_Customer.max().date()),
      'comparacion_fecha_inicio':str(df_cur.Dt_Customer.min().date()),
      'semilla':CFG['semilla'], 'alpha':CFG['alpha'],
      'adwin_delta':CFG['adwin_delta'], 'tareas':'Caso HyM: Data Drift',
      'advertencia':'Las fechas son de incorporacion de clientes, no de compras',
    }
    (RUTA_RESULTADOS/'manifiesto.json').write_text(
        json.dumps(manifiesto,indent=2,ensure_ascii=False),encoding='utf-8')
    ruta_zip=shutil.make_archive(str(RUTA_RESULTADOS.parent/'HM_Data_Drift_Resultados'),'zip',RUTA_RESULTADOS)
print(f'Listo. ZIP de evidencias: {ruta_zip}')
print('En Colab, opcional: from google.colab import files; files.download(ruta_zip)')

### 12.4. Interpretación integrada entre métodos

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Redactar evidencias reales señalando convergencias, divergencias y variables más alteradas; evitar contar alertas de ADWIN como valores p o PSI como prueba de significancia.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Tres hallazgos empíricos con cifras extraídas de las tablas y un apartado de limitaciones, completados después de ejecutar.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Espacio para Marco — rellenar al ejecutar datos reales:**

1. **Hallazgo numérico más importante:** variable, n de cada cohorte, D y p ajustado de KS, PSI e intensidad.
2. **Hallazgo categórico más importante:** variable, variación en puntos porcentuales, chi-cuadrado y p ajustado **solo si** se cumplen supuestos.
3. **Hallazgo ADWIN:** número de alertas, fechas/índices observados y conexión *cautelosa* con el corte.
4. **Comparación:** al menos un acuerdo y una discrepancia entre KS, PSI, chi-cuadrado y ADWIN, según aplique.
5. **Límites:** no hay fecha individual de compra, no existe etiqueta de producción ni ground truth de drift real; p múltiple corregido, resultados dependientes de cohorte y configuración.

## 13. Conclusiones, bibliografía y entrega

**Responsable principal:** Marco. **Objetivo:** cerrar un informe verificable, sin resultados anticipados.

### 13.1. Conclusiones y recomendaciones

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Redactar respuesta a cada objetivo, implicaciones razonables y límites; no atribuir causalidad ni medir desempeño de recomendaciones si no se entrenó un modelo.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Conclusiones con cifras verificables y delimitación entre drift real/experimental.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Plantilla para redactar al completar el análisis:**

- **Conclusión 1 · Datos:** indicar qué características cambiaron entre cohortes, sustentando con resultados.
- **Conclusión 2 · Métodos:** indicar qué métodos coincidieron y en qué difirieron, sin afirmar equivalencias estadísticas.
- **Conclusión 3 · ADWIN:** indicar si emitió alertas, dónde y con qué límites de interpretación.
- **Conclusión 4 · Control:** demostrar qué modificaciones fueron inyectadas y qué métodos las detectaron.
- **Limitaciones:** muestra transversal, fecha de alta y no de compra, variables sin historial de versiones, supuestos de pruebas, multiplicidad y sensibilidad a parámetros.
- **Recomendación futura:** si se obtiene un histórico real de transacciones fechado, repetir el monitoreo sobre observaciones agregadas por período.

**Estado:** pendiente de completar con los resultados de ejecución reales. No sustituir por cifras supuestas.

### 13.2. Fuentes y referencias reproducibles

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Citar notebooks entregados por el profesor y documentación oficial de técnicas utilizadas; registrar fecha de consulta y nombre del CSV de entrada.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Fuentes identificables con enlaces oficiales y distinción entre material docente y documentación técnica.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

**Material proporcionado por el docente:**

- `data-drift VFinal.ipynb` (funciones y ejemplos de KS, PSI y chi-cuadrado).
- `analisis datadrift .ipynb` (notebook de análisis de campaña de marketing).
- `marketing_campaign.csv` (archivo de entrada, entregar o referenciar su ubicación en Colab).

**Documentación técnica:**

- [River: drift.ADWIN](https://riverml.xyz/latest/api/drift/ADWIN/) y [código oficial en GitHub](https://github.com/online-ml/river/blob/main/river/drift/adwin.py).
- [SciPy: ks_2samp](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.ks_2samp.html).
- [SciPy: chi2_contingency](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.chi2_contingency.html).
- [Pandas: read_csv](https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html).

**Nota:** los umbrales PSI `<0.10`, `<0.25` y `≥0.25` son reglas heurísticas y no equivalen a pruebas de hipótesis.

### 13.3. Checklist final de entrega para los cinco integrantes

**Encargado: Marco.**

**Tarea exacta para el integrante/agente:** Efectuar ejecución limpia de arriba abajo, corregir errores, preservar conclusiones humanas y validar que no quedaron resultados sintéticos tratados como reales.

**Entradas y dependencias:** Ninguna adicional.

**Criterio de aceptación verificable:** Todas las casillas se revisan manualmente antes de enviar el .ipynb final.

> **Regla de integración:** editar únicamente las celdas de esta subsección o añadir celdas inmediatamente después; no renombrar funciones, contratos ni variables compartidas. Entregar explicación, evidencia y código reproducible.

### Lista de aceptación (revisar manualmente)

- [ ] **Poma:** CSV cargado una sola vez; tipo de fecha correcto; calidad, preprocesamiento y EDA verificados.
- [ ] **Marco:** cohortes sin solapamientos y con fechas explicadas; semilla, entorno y parámetros consignados.
- [ ] **Vera:** KS y PSI calculados sobre las mismas cohortes; p ajustados, bins de referencia y advertencias registradas.
- [ ] **Aldana:** chi-cuadrado con chequeo de esperados; inyección artificial no destructiva; comparación controlada explicada.
- [ ] **Brickzon:** ADWIN ejecutado con barra de progreso y alertas trazables; límites de interpretación expuestos.
- [ ] **Marco:** resumen integrado, conclusiones no inventadas, exports generados y ejecución `Entorno de ejecución → Ejecutar todo` en Colab.
- [ ] Los cinco han incluido **explicación + código + resultados + interpretación** en sus subsecciones.
- [ ] No se publicaron secretos, tokens, rutas personales ni datos identificables innecesarios.

> **Instrucción final para agentes:** respetar los contratos definidos en 2.3; al terminar, comunicar qué celdas cambiaste, qué ejecutaste, duración aproximada y cualquier fallo reproducible. No declarar PASS sin ejecutar los tests pertinentes.